# 📂 Notebook 03: Clustering Probabilístico de Temas (LDA)
**Proyecto:** Análisis de Sentimiento Bayesiano

Siguiendo las recomendaciones del profesorado, implementamos un modelo de **Latent Dirichlet Allocation (LDA)** para realizar un clustering suave de los mensajes. 

**Objetivos:**
1. Entrenar un modelo LDA sobre la matriz TF-IDF.
2. Extraer la distribución de temas ($\theta$) para cada reseña (distribución de Dirichlet).
3. Visualizar los temas extraídos para validar la coherencia semántica.
4. Generar la señal adicional que alimentará a la BNN.

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [1]:
import os
import sys

# 1. AJUSTE DE RUTA (DEBE IR PRIMERO)
# Si estamos en la carpeta 'notebooks', retrocedemos a la raíz para ver 'src'
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

# Añadimos la raíz al path por si acaso (muy útil en macOS)
root_path = os.getcwd()
if root_path not in sys.path:
    sys.path.append(root_path)

# 2. AHORA SÍ, LAS IMPORTACIONES DEL PROYECTO
import joblib
import numpy as np
from src.data.loader import load_tfidf_splits
from src.models.lda import TopicModeler

# Cargamos los datos para LDA (TF-IDF)
X_train_tfidf, X_val_tfidf, X_test_tfidf, labels = load_tfidf_splits()

print(f"✅ Directorio actual: {os.getcwd()}")
print(f"✅ Dimensiones de entrenamiento: {X_train_tfidf.shape}")

✅ Directorio actual: /Users/jimenamonteagudoruiz/Bayesian-Sentiment-Analysis
✅ Dimensiones de entrenamiento: (7999, 10000)


In [2]:
# Definimos el número de temas (K). Simón sugirió que el análisis da para más.
# Probaremos con K=10 para capturar diferentes matices (guion, actuación, efectos, etc.)
N_TOPICS = 10
lda_modeler = TopicModeler(n_components=N_TOPICS)

# Entrenamos el modelo LDA
lda_modeler.fit(X_train_tfidf)

# Guardamos el modelo para usarlo en el pipeline final y en la app
os.makedirs("experiments/results", exist_ok=True)
lda_modeler.save("experiments/results/lda_model.pkl")
print("✅ Modelo LDA entrenado y guardado.")

✅ Modelo LDA entrenado y guardado.


In [3]:
# Extraemos theta (la distribución de probabilidad sobre temas) para el test set
theta_test = lda_modeler.get_topics(X_test_tfidf)

# Verificamos que sea una distribución de probabilidad (debe sumar 1 por fila)
sample_idx = 0
sum_prob = np.sum(theta_test[sample_idx])
print(f"Suma de probabilidades del ejemplo {sample_idx}: {sum_prob:.4f}")
print(f"Distribución de temas (θ): \n{theta_test[sample_idx]}")

# Esto cumple con el requisito de usar modelos de Dirichlet para clustering suave.

Suma de probabilidades del ejemplo 0: 1.0000
Distribución de temas (θ): 
[0.01209905 0.01209905 0.01209905 0.01209905 0.01209905 0.01209905
 0.89110857 0.01209905 0.01209905 0.01209905]


In [4]:
# Función para visualizar las palabras clave de cada tema
def print_top_words(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        message = f"Tema #{topic_idx}: "
        message += " ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)

from src.data.loader import load_vectorizer
vectorizer = load_vectorizer()
print_top_words(lda_modeler.lda, vectorizer.get_feature_names_out())

Tema #0: croc ocean zeta zeta jones worst 12 clooney just stuff designed
Tema #1: darth frye vader vampire bat lionel atwill atwill wray fay wray ewoks stormtroopers
Tema #2: streep meryl meryl streep dustin hoffman dustin agency client impressive movie good sneak
Tema #3: hawn abbott abbott costello lion king costello goldie television series bergman choreography lion
Tema #4: modesty blaise modesty blaise alexandra quentin title role depend foggy film boring commitment
Tema #5: paltrow gwyneth gwyneth paltrow emma like better parallels jane austen kate accessible things like
Tema #6: movie film like just good time really story bad great
Tema #7: soylent soylent green heston robinson 1973 understand people shares admired thorne really make
Tema #8: busey gary busey rosario rooney gather brody wastes rourke dawson pursues
Tema #9: documentary davies taylor people story texas film public race rooney
